# ADMET-Engine v3.0 (*CardioPhys-3D*) — WP4 & WP5: Knowledge Distillation & Closed-Loop Molecular Repair

This notebook executes **Säule 3 (WP4 + WP5)** of the CardioPhys-3D Master-Paper:
1. **WP4 (Knowledge Distillation):** Compresses the heavy SE(3)-equivariant 3D PaiNN teacher (`models/cardiophys_3d_v3.pt`) into a sub-2 ms edge student (`CardioPhysStudentNet`) via closed-form Gaussian KL divergence + censored Tobit NLL.
2. **End-to-End Coupling:** Connects `SMILES -> Student 3-Channel pIC50 -> Differentiable O'Hara-Rudy CiPA ODE -> qNet / APD90 / P(TdP)`.
3. **WP5 (Closed-Loop Molecular Repair Agent):** Automatically detoxifies withdrawn/high-risk cardiotoxic drugs (**Terfenadine**, **Astemizole**, **Cisapride**) via curated Matched Molecular Pair (MMP) medicinal chemistry edits while preserving scaffold similarity (Tanimoto >= 0.55), QED, SA-Score <= 4.8, and 0 PAINS alerts.

In [ ]:
# 1. Clone / update repository & install dependencies
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/content/Admet-Engine')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/JakobFerschin/Admet-Engine.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'scipy', 'pandas', 'scikit-learn', 'lightgbm', 'xgboost'], check=True)

import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 2. Optional: Mount Google Drive to load WP1/WP3 artifacts and save WP4/WP5 outputs
from pathlib import Path
import shutil

DRIVE_DIR = Path('/content/drive/MyDrive/CardioPhys3D_v3')
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    for fname, dest in [
        ('v3_multichannel_chembl37.csv', Path('data/v3_multichannel_chembl37.csv')),
        ('v3_conformers_3d.pt', Path('data/v3_conformers_3d.pt')),
        ('cardiophys_3d_v3.pt', Path('models/cardiophys_3d_v3.pt')),
    ]:
        src = DRIVE_DIR / fname
        if src.exists() and not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest)
            print(f'Restored {fname} from Drive -> {dest}')
except Exception as e:
    print('Drive mount skipped:', e)

In [ ]:
# 3. WP4: Distill 3D PaiNN Teacher -> Fast Edge Student (<2 ms/mol)
import json, subprocess, sys
from pathlib import Path

teacher_ckpt = Path('models/cardiophys_3d_v3.pt')
data_csv = Path('data/v3_multichannel_chembl37.csv')
conf_pt = Path('data/v3_conformers_3d.pt')

if teacher_ckpt.exists() and data_csv.exists() and conf_pt.exists():
    subprocess.run([
        sys.executable, 'src/training/distill_wp4.py',
        '--data-csv', str(data_csv),
        '--conformers-pt', str(conf_pt),
        '--teacher-ckpt', str(teacher_ckpt),
        '--epochs', '50',
        '--batch-size', '128',
        '--distill-weight', '0.55',
    ], check=True)
    metrics = json.loads(Path('models/cardiophys_student_v3_metrics.json').read_text())
    print(json.dumps(metrics, indent=2))
else:
    print('WP3 teacher checkpoint not found yet — run cardiophys_3d_wp3_gpu.ipynb first to distill from the 3D teacher.')
    print('Proceeding to WP5 Closed-Loop Molecular Repair using the trained ChEMBL v37 hERG ensemble oracle.')

In [ ]:
# 4. WP5: Closed-Loop Molecular Repair Benchmark (Terfenadine, Astemizole, Cisapride)
import sys
from pathlib import Path
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem import Draw

sys.path.insert(0, str(Path('src').resolve()))

from models.chembl_ensemble import ChEMBLADMETEnsemble
from models.distillation import CardioPhysEndToEndPipeline, CardioPhysStudentNet
from physiology.cipa28 import get_cipa28_drug
from repair.molecular_repair import ClosedLoopRepairAgent

student_ckpt = Path('models/cardiophys_student_v3.pt')
if student_ckpt.exists():
    print('Using distilled WP4 -> WP2 ORd-CiPA ODE P(TdP) oracle!')
    ckpt = torch.load(str(student_ckpt), map_location='cpu', weights_only=False)
    cfg = ckpt['config']
    student = CardioPhysStudentNet(**cfg)
    student.load_state_dict(ckpt['model_state_dict'])
    pipe = CardioPhysEndToEndPipeline(student=student, device='cuda' if torch.cuda.is_available() else 'cpu')
    oracle_name = 'CardioPhys-3D Distilled Student -> ORd-CiPA P(TdP)'
    def risk_oracle(smiles_list):
        preds = pipe.predict_smiles(smiles_list, conc_uM=1.0)
        return [p.tdp_risk_score if p is not None else 1.0 for p in preds]
else:
    print('Using trained v2.0 ChEMBL v37 4-family hERG ensemble oracle!')
    ens = ChEMBLADMETEnsemble(bundle_dir=Path('models'), device='cpu')
    oracle_name = 'ChEMBL v37 Ensemble P(hERG active)'
    def risk_oracle(smiles_list):
        mols = [Chem.MolFromSmiles(s) for s in smiles_list]
        preds = ens.predict_mols(mols)
        return [float(p['predictions']['hERG_inhibition']) for p in preds]

agent = ClosedLoopRepairAgent(
    risk_oracle=risk_oracle,
    min_tanimoto=0.55,
    max_sa_score=4.8,
    min_qed=0.25,
    require_zero_pains=True,
)

targets = ['Terfenadine', 'Astemizole', 'Cisapride']
rows = []
grid_mols, grid_legends = [], []

for name in targets:
    drug = get_cipa28_drug(name)
    rep = agent.repair(drug.smiles, n_rounds=2, beam_width=5, top_k=3)
    seed = rep['seed']
    best = rep['repaired_candidates'][0]
    rows.append({
        'Drug': name,
        'Oracle': oracle_name,
        'Seed Risk': seed['risk_score'],
        'Repaired Risk': best.risk_score,
        'Delta Risk': best.delta_risk,
        'Tanimoto': best.tanimoto,
        'Seed QED -> New QED': f"{seed['qed']} -> {best.qed}",
        'SA-Score': best.sa_score,
        'cLogP': f"{seed['clogp']} -> {best.clogp}",
        'TPSA': f"{seed['tpsa']} -> {best.tpsa}",
        'Edit Path': ' -> '.join(best.edit_path),
        'Repaired SMILES': best.smiles,
    })
    grid_mols.extend([Chem.MolFromSmiles(seed['smiles']), Chem.MolFromSmiles(best.smiles)])
    grid_legends.extend([
        f"{name} (Risk={seed['risk_score']:.3f})",
        f"{name} Repaired (Risk={best.risk_score:.3f}, d={best.delta_risk:+.3f})",
    ])

df_res = pd.DataFrame(rows)
display(df_res)
img = Draw.MolsToGridImage(grid_mols, molsPerRow=2, subImgSize=(420, 280), legends=grid_legends)
display(img)

In [ ]:
# 5. Backup WP4/WP5 outputs to Google Drive
import shutil
from pathlib import Path

df_res.to_csv('models/wp5_repair_benchmark.csv', index=False)
if DRIVE_DIR.exists():
    for f in ['models/cardiophys_student_v3.pt', 'models/cardiophys_student_v3_metrics.json', 'models/wp5_repair_benchmark.csv']:
        p = Path(f)
        if p.exists():
            shutil.copy2(p, DRIVE_DIR / p.name)
            print(f'Saved {p.name} -> {DRIVE_DIR / p.name}')